In [153]:
from utilsforecast.preprocessing import fill_gaps
from tinyshift.stats import remove_leading_zeros, is_obsolete
from tinyshift.plot import stationarity_analysis, pami, residual_analysis, seasonal_decompose
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from statsmodels.tsa.seasonal import MSTL
from statsforecast.models import SeasonalNaive, AutoETS
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, bias, cfe
from tinyshift.modelling import DMSTLWrapper, fourier_seasonality
from mlforecast.utils import PredictionIntervals
from tinyshift.series import (
    wape,
    pbias,
    score,
    forecast_instability,
    detect_seasonal_periods,
    extract_mstl_components,
    fva_rmae,
    mach,
    permutation_auto_mutual_information,
)

from utilsforecast.preprocessing import fill_gaps

In [2]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = df.groupby("unique_id")[df.columns].apply(remove_leading_zeros).reset_index(drop=True)
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180
obsolete_series = df.groupby("unique_id")[df.columns].apply(is_obsolete, days_obsoletes)
obsolote_ids = obsolete_series[obsolete_series].index.tolist()
assert len(obsolote_ids) == 0, f"Obsolete series found: {obsolote_ids}"

In [4]:
df.isnull().sum()

unique_id      0
ds             0
y              0
monthly_sin    0
monthly_cos    0
dtype: int64

In [144]:
from pathlib import Path
from scipy.signal import find_peaks


def first_pami_local_minimum(values, max_tau=365, m=3, delay=1):
    values = np.asarray(values, dtype=float)
    max_valid_tau = len(values) - (m - 1) * delay - 1
    taus = np.arange(1, min(max_tau, max_valid_tau) + 1)
    pami_values = np.array([
        permutation_auto_mutual_information(
            values, tau=int(tau), m=m, delay=delay, normalize=True
        )
        for tau in taus
    ])
    minima, _ = find_peaks(-pami_values)
    position = minima[0] if len(minima) else int(np.argmin(pami_values))
    return int(taus[position]), float(pami_values[position])


# Online Retail II: mesmo painel diário e os mesmos 12 SKUs do demand_class.
dataset_dir = Path(kagglehub.dataset_download("mashlyn/online-retail-ii-uci"))
retail_files = list(dataset_dir.rglob("*.csv"))
raw_retail = pd.read_csv(retail_files[0], encoding="latin1")
price_col = "Price" if "Price" in raw_retail.columns else "UnitPrice"
retail = raw_retail[["StockCode", "InvoiceDate", "Quantity", price_col]].copy()
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail["StockCode"] = retail["StockCode"].astype(str)
retail = retail.loc[
    retail["InvoiceDate"].notna()
    & (retail["Quantity"] > 0)
    & (retail[price_col] > 0)
].copy()
retail["ds"] = retail["InvoiceDate"].dt.floor("D")
retail = retail.rename(columns={"StockCode": "unique_id"})
benchmark_skus = (
    retail.groupby("unique_id")["Quantity"]
    .sum()
    .nlargest(12)
    .index
    .tolist()
)
dates = pd.date_range(retail["ds"].min(), retail["ds"].max(), freq="D")
benchmark_values = (
    retail.groupby(["ds", "unique_id"])["Quantity"]
    .sum()
    .rename("y")
    .reset_index()
    
)
retail = pd.merge(benchmark_values, retail, on=["ds", "unique_id"], how="left")

In [146]:
benchmark_panel = retail.copy()
benchmark_panel["y"] = benchmark_panel["y"].fillna(0.0)
benchmark_cutoff = benchmark_panel["ds"].quantile(0.80)
benchmark_train = benchmark_panel[benchmark_panel["ds"] <= benchmark_cutoff].copy()
benchmark_test = benchmark_panel[benchmark_panel["ds"] > benchmark_cutoff].copy()
benchmark_horizon = 28

In [150]:
benchmark_panel = benchmark_panel.drop(columns=["InvoiceDate", "Quantity"])

In [ ]:
benchmark_panel

fill_gaps(benchmark_panel, freq="D", end="per_serie", id_col="unique_id", time_col="ds")

,ds,unique_id,y,Price
0,2009-12-01,10002,12,0.85
1,2009-12-01,10120,60,0.21
2,2009-12-01,10123C,3,1.30
3,2009-12-01,10123G,2,1.70
4,2009-12-01,10125,5,1.70
...,...,...,...,...
1041666,2011-12-09,POST,10,18.00
1041667,2011-12-09,POST,10,18.00
1041668,2011-12-09,POST,10,18.00
1041669,2011-12-09,POST,10,18.00


ValueError: cannot handle a non-unique multi-index!

In [151]:
def residual_model_callable(nlags, freq):
    return MLForecast(
        models=[RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1)],
        freq=freq,
        lags=nlags,
        date_features=["dayofweek"]
    )


dmstl = DMSTLWrapper(
    residual_model_callable=residual_model_callable,
    freq="D",
    seasonal_detection_params={"top_k": 2, "fallback": [7]},
    nlags="auto",
    pami_params={"max_tau": 30, "m": 3, "delay": 1, "return_mode": "range", "fallback": 1},
)
dmstl.fit(benchmark_train, target_col="y")

ValueError: Series contain missing or duplicate timestamps with the specified freq D
Affected series: ['10002']
Consider using the fill_gaps parameter or preprocessing your data.

In [115]:
result = dmstl.predict(h=benchmark_horizon)
result = benchmark_test.merge(
    result[["unique_id", "ds", "RandomForestRegressor"]],
    on=["unique_id", "ds"],
)
result["RandomForestRegressor"] = result["RandomForestRegressor"].clip(lower=0)

In [116]:
metrics = [rmse, mae, bias, wape, pbias, score, forecast_instability]

In [117]:
evaluate(result, 
         metrics=[forecast_instability], 
         models=["RandomForestRegressor"], 
         id_col="unique_id", 
         time_col="ds", 
         target_col="y").sort_values(by="RandomForestRegressor", ascending=False)

,unique_id,metric,RandomForestRegressor
2,21212,forecast_instability,203.273910
4,22197,forecast_instability,191.610198
5,84879,forecast_instability,176.762302
1,85099B,forecast_instability,157.136641
11,22492,forecast_instability,147.644185
0,84077,forecast_instability,144.148148
10,84991,forecast_instability,143.494809
9,21977,forecast_instability,137.001644
7,23166,forecast_instability,70.324108
3,85123A,forecast_instability,57.955816
